# Train imagejev on Kaggle

Run this notebook with a **GPU** (T4) and **Internet** on. Attach two inputs:

1. the dataset built with `package_dataset` (default name `imagejev-data`);
2. from the second session on, the **output of the previous version of this notebook** (its slug is `imagejev-train` below), so training resumes.

Each session trains until the time budget in `configs/kaggle_v0.yaml`, saves `last.pt` and stops. Then use **Save Version -> Save & Run All (Commit)** so the output is kept for the next session. See `docs/kaggle.md`.


In [ ]:
!git clone --depth 1 https://github.com/guangshinhaha/imagejev.git
%cd imagejev
!pip -q install -e ".[model]"

In [ ]:
import os
import subprocess

print(
    subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    or "no GPU: switch the accelerator to a GPU"
)
DATA = "/kaggle/input/imagejev-data"
assert os.path.isdir(DATA), f"attach the dataset first: {DATA} not found"
print(sorted(os.listdir(DATA)))

In [ ]:
from imagejev.train.kaggle import restore_run

PREVIOUS = "/kaggle/input/imagejev-train/runs/v0"  # output of the previous notebook version
RUN = "/kaggle/working/runs/v0"
print("restored a previous checkpoint" if restore_run(PREVIOUS, RUN) else "starting a fresh run")

In [ ]:
!python -m imagejev.train.loop --config configs/kaggle_v0.yaml

In [ ]:
import csv
import os

run = "/kaggle/working/runs/v0"
print(sorted(os.listdir(run)))
rows = (
    list(csv.DictReader(open(f"{run}/select_log.csv")))
    if os.path.exists(f"{run}/select_log.csv")
    else []
)
print(
    "val macro log loss by step:", [(r["step"], round(float(r["macro_log_loss"]), 3)) for r in rows]
)
last = list(csv.DictReader(open(f"{run}/train_log.csv")))[-1]
print("last training step:", last["step"])

If the last step is below the configured total, **Save Version** and start another session; it resumes automatically.
